# Modül 01: İlk PyTorch Modeli
### Doğrusal Regresyon ile İlk Model

---

**Proje:** Eğitim Teknolojileri için PyTorch Eğitimi  
**Hazırlayan:** Uğur Sırvermez — Bursa Uludağ Üniversitesi  
**Lisans:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)  
**Repo:** [github.com/ugursirvermez/PyTorch_Education](https://github.com/ugursirvermez/PyTorch_Education)

> Bu modül, [Daniel Bourke](https://github.com/mrdbourke)'ün  
> **[Zero to Mastery Learn PyTorch for Deep Learning](https://github.com/mrdbourke/pytorch-deep-learning)**  
> ([learnpytorch.io](https://www.learnpytorch.io)) adlı çalışmasından ilham alınarak hazırlanmıştır.  
> İçerik Türkçeye çevrilmiş, eğitim teknolojistleri için yeniden yapılandırılmış ve pedagojik modül formatına dönüştürülmüştür.

---

## Ogrenme Hedefleri

- `nn.Module` sınıfından türetilmiş basit bir doğrusal regresyon modeli tanımlayabilirsiniz.
- Eğitim verisi ile test verisini ayırabilir (train/test split) ve bunun önemini açıklayabilirsiniz.
- Kayıp fonksiyonu (Loss Function) ve optimize edici (Optimizer) seçip modelinizi eğitebilirsiniz.
- Eğitim döngüsü adımlarını (forward → loss → backward → step) sırasıyla uygulayabilirsiniz.
- Modelin tahminlerini görselleştirebilirsiniz.

---

## Ön Koşullar

Modül 00: PyTorch'a Giriş tamamlanmış olmalıdır. Tensör kavramına ve Python sınıflarına (class) temel düzeyde aşinalık gereklidir.

---

⏱ Tahmini süre: 2.5–3 saat

---

## Modül Özeti

Bu modülde ilk gerçek PyTorch modelinizi sıfırdan kuruyorsunuz. Doğrusal regresyon üzerinden **model → kayıp → geri yayılım → güncelleme** döngüsünü kavrayarak tüm derin öğrenme eğitiminin temel ritmine giriyorsunuz.

---


<a href="https://colab.research.google.com/github/ugursirvermez/PyTorch_Education/blob/main/pytorch_first_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## Bölüm 1 — Kütüphaneler

PyTorch modellemesi için gerekli kütüphaneleri **tek bir hücrede** tanımlıyoruz.
Bunu bir kez yapmanız yeterli; sonraki hücrelerde tekrar `import` yazmanıza gerek yok.

- `torch` → Tensörler ve model işlemleri
- `torch.nn` → Sinir ağı katmanları ve kayıp fonksiyonları
- `matplotlib.pyplot` → Görselleştirme
- `numpy` → Sayısal dizi işlemleri (grafik çizmek için tensör dönüşümünde kullanılır)
- `pathlib.Path` → Platformdan bağımsız dosya yolu yönetimi


In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print(f"PyTorch sürümü: {torch.__version__}")


---

## Bölüm 2 — Cihaz Seçimi (CPU / GPU)

Derin öğrenme hesaplamaları GPU'da onlarca kat daha hızlı çalışır.
Aşağıdaki **agnostik cihaz kodu** sayesinde modeliniz GPU varsa otomatik olarak
GPU'ya, yoksa CPU'ya yönlendirilir. Bu kalıbı tüm projelerinizde kullanabilirsiniz.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan cihaz: {device}")


---

## Bölüm 3 — Veri Hazırlama

Gerçek bir veri seti kullanmadan önce **sentetik (yapay) veri** üretelim.
Doğrusal regresyon formülü şöyle: `y = ağırlık × X + sapma`

Burada `ağırlık = 0.7` ve `sapma = 0.3` olarak belirliyoruz.
Model eğitim sırasında bu değerleri tahmin etmeye çalışacak.

> 💡 **Neden yapay veri?** Doğru cevabı (ağırlık ve sapma) biz belirlediğimiz için
> modelin ne kadar iyi öğrendiğini kolayca ölçebiliriz.


In [ ]:
# Doğrusal regresyon için parametreler
agirlik = 0.7
sapma   = 0.3

# X veri aralığı oluşturma
X = torch.arange(0, 1, 0.02).unsqueeze(dim=1)
y = agirlik * X + sapma

print(f"X boyutu : {X.shape}")
print(f"y boyutu : {y.shape}")
print(f"İlk 5 X : {X[:5].squeeze()}")
print(f"İlk 5 y : {y[:5].squeeze()}")


---

## Bölüm 4 — Eğitim / Test Bölme

Modeli **gördüğü verilerle** değerlendirmek yanıltıcı olur.
Bu nedenle veriyi ikiye ayırıyoruz:

- **Eğitim seti (%80)** → Modelin öğreneceği veriler
- **Test seti (%20)** → Modelin hiç görmediği veriler (gerçek performans ölçümü)

Bu ayrım, modelin gerçek dünya verilerine ne kadar genelleyebildiğini ölçmemizi sağlar.


In [ ]:
# %80 eğitim, %20 test
train_split = int(0.8 * len(X))

X_train, y_train = X[:train_split], y[:train_split]
X_test,  y_test  = X[train_split:],  y[train_split:]

print(f"Eğitim örnekleri : {len(X_train)}")
print(f"Test örnekleri   : {len(X_test)}")


---

## Bölüm 5 — Görselleştirme Yardımcı Fonksiyonu

Eğitim verimizi, test verimizi ve modelin tahminlerini tek bir grafik üzerinde
karşılaştırmak için bir yardımcı fonksiyon tanımlıyoruz.

Bu fonksiyon bu modül boyunca tekrar tekrar kullanılacak.


In [ ]:
def tahminleri_goster(
    egitim_verisi, egitim_etiketleri,
    test_verisi,   test_etiketleri,
    tahminler=None
):
    """Eğitim, test ve tahmin verilerini karşılaştırmalı olarak çizer."""
    plt.figure(figsize=(10, 6))
    plt.scatter(egitim_verisi, egitim_etiketleri,
                c="steelblue", s=10, label="Eğitim Verisi")
    plt.scatter(test_verisi, test_etiketleri,
                c="seagreen", s=10, label="Test Verisi")
    if tahminler is not None:
        plt.scatter(test_verisi, tahminler,
                    c="tomato", s=10, label="Tahminler")
    plt.xlabel("X")
    plt.ylabel("y")
    plt.title("Eğitim / Test / Tahmin Karşılaştırması")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Eğitim ve test verisini görselleştir (henüz tahmin yok)
tahminleri_goster(X_train, y_train, X_test, y_test)


---

## Bölüm 6 — Model Tanımlama

PyTorch'ta modeller `nn.Module` sınıfından türetilir.
Doğrusal regresyon için `nn.Linear` kullanıyoruz; bu katman `y = Wx + b` hesabını
otomatik olarak yapar ve ağırlıkları (W, b) `nn.Parameter` olarak tutar.

```
       X (giriş)
          │
    ┌─────▼──────┐
    │ nn.Linear  │  ← W ve b burada öğrenilir
    └─────┬──────┘
          │
       y (tahmin)
```


In [ ]:
class DogrusalRegressionModeli(nn.Module):
    """Tek giriş — tek çıkış doğrusal regresyon modeli.

    nn.Linear(in_features=1, out_features=1) kullanır;
    ağırlık (weight) ve sapma (bias) parametrelerini otomatik yönetir.
    """

    def __init__(self):
        super().__init__()
        self.lineer_katman = nn.Linear(in_features=1, out_features=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.lineer_katman(x)


# Tekrarlanabilirlik için sabit tohum (seed)
torch.manual_seed(42)
model = DogrusalRegressionModeli()
model.to(device)

print("Model parametreleri (başlangıç — rastgele):")
print(model.state_dict())


---

## Bölüm 7 — Kayıp Fonksiyonu ve Optimizer

Modelin ne kadar yanıldığını ölçmek için **kayıp fonksiyonu**,
bu yanılgıyı azaltmak için ise **optimizer** kullanıyoruz.

| Bileşen | Kullanılan | Açıklama |
|---|---|---|
| Kayıp fonksiyonu | `L1Loss` (MAE) | Tahmin ile gerçek arasındaki ortalama mutlak fark |
| Optimizer | `SGD` | Stokastik Gradyan İnişi; `lr` parametresi öğrenme hızını belirler |

> 💡 **Öğrenme hızı (lr)** kritik bir hiperparametredir.
> Çok büyük → model ıraksayabilir; çok küçük → öğrenme çok yavaş olur.


In [ ]:
kayip_fn  = nn.L1Loss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

print(f"Kayıp fonksiyonu : {kayip_fn}")
print(f"Optimizer        : {optimizer}")


---

## Bölüm 8 — Eğitim Döngüsü

Derin öğrenmede eğitim döngüsü her **epoch** (devir) için şu 5 adımı uygular:

```
1. model.train()       → Eğitim modunu aç (dropout, batchnorm aktif)
2. y_tahmin = model(X) → İleri geçiş (Forward pass)
3. loss = kayip_fn(…)  → Kaybı hesapla
4. loss.backward()     → Geri yayılım (gradyanları hesapla)
5. optimizer.step()    → Ağırlıkları güncelle
```

Her 10 epoch'ta bir test kaybını da hesaplayarak eğitim sürecini izliyoruz.


In [ ]:
# Verileri seçilen cihaza taşı
X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)

# Kayıp değerlerini kaydet
epoch_kayitlari     = []
egitim_kayiplari    = []
test_kayiplari      = []

EPOCH_SAYISI = 200

for epoch in range(EPOCH_SAYISI):

    # --- Eğitim ---
    model.train()
    y_tahmin = model(X_train)
    kayip = kayip_fn(y_tahmin, y_train)
    optimizer.zero_grad()
    kayip.backward()
    optimizer.step()

    # --- Değerlendirme ---
    model.eval()
    with torch.inference_mode():
        test_tahmini = model(X_test)
        test_kayip   = kayip_fn(test_tahmini, y_test)

    # Her 10 epoch'ta raporla
    if epoch % 10 == 0:
        epoch_kayitlari.append(epoch)
        egitim_kayiplari.append(kayip.item())
        test_kayiplari.append(test_kayip.item())
        print(f"Epoch {epoch:3d} | Eğitim kaybı: {kayip.item():.4f} "
              f"| Test kaybı: {test_kayip.item():.4f}")

print("\nEğitim tamamlandı!")
print(f"Öğrenilen parametreler: {model.state_dict()}")


---

## Bölüm 9 — Kayıp Eğrilerini Görselleştirme

Eğitim ve test kayıplarının epoch'lara göre grafiğini çizmek,
modelin ne zaman öğrenmeyi bıraktığını veya aşırı öğrenmeye başladığını
görmenin en hızlı yoludur.

- İki eğri birlikte azalıyorsa → model öğreniyor ✅
- Eğitim kaybı azalırken test kaybı artıyorsa → aşırı öğrenme (overfitting) ⚠️


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(epoch_kayitlari, egitim_kayiplari, label="Eğitim Kaybı", color="steelblue")
plt.plot(epoch_kayitlari, test_kayiplari,   label="Test Kaybı",   color="tomato")
plt.xlabel("Epoch")
plt.ylabel("Kayıp (L1 / MAE)")
plt.title("Eğitim ve Test Kayıp Eğrileri")
plt.legend()
plt.tight_layout()
plt.show()


---

## Bölüm 10 — Tahminleri Görselleştirme

Eğitim sonrası modelin test verisi üzerindeki tahminlerini
Bölüm 5'te tanımladığımız `tahminleri_goster` fonksiyonu ile çizelim.


In [ ]:
model.eval()
with torch.inference_mode():
    y_tahminler = model(X_test)

# CPU'ya al (matplotlib için gerekli)
tahminleri_goster(
    X_train.cpu(), y_train.cpu(),
    X_test.cpu(),  y_test.cpu(),
    tahminler=y_tahminler.cpu()
)


---

## Bölüm 11 — Model Kaydetme

PyTorch'ta modeli kaydetmenin önerilen yolu `state_dict()` kullanmaktır.
`state_dict`, modelin ağırlıklarını ve parametrelerini bir Python sözlüğü olarak saklar;
bu sayede sadece ağırlıkları kaydedip daha sonra aynı mimariye yükleyebilirsiniz.

```
state_dict → { "lineer_katman.weight": tensor(...),
               "lineer_katman.bias"  : tensor(...) }
```


In [ ]:
MODEL_KLASORU = Path("models")
MODEL_KLASORU.mkdir(parents=True, exist_ok=True)

MODEL_ADI  = "01_dogrusal_regresyon_modeli.pth"
MODEL_YOLU = MODEL_KLASORU / MODEL_ADI

torch.save(obj=model.state_dict(), f=MODEL_YOLU)
print(f"Model kaydedildi → {MODEL_YOLU}")


---

## Bölüm 12 — Model Yükleme

Kaydedilen `state_dict`'i geri yüklemek için önce **aynı mimariye** sahip boş bir
model oluşturup sonra ağırlıkları bu modele yüklüyoruz.


In [ ]:
# Boş model oluştur (mimari aynı olmalı)
yuklenen_model = DogrusalRegressionModeli()
yuklenen_model.load_state_dict(torch.load(MODEL_YOLU))
yuklenen_model.to(device)

print("Yüklenen model parametreleri:")
print(yuklenen_model.state_dict())

# Yüklenen modelin orijinal modelle aynı tahminleri ürettiğini doğrula
yuklenen_model.eval()
with torch.inference_mode():
    yuklenen_tahmin = yuklenen_model(X_test)

print(f"\nTahminler eşleşiyor mu: {torch.allclose(y_tahminler, yuklenen_tahmin)}")


---

## Modül Sonu — Özet

Bu modülde ilk uçtan uca PyTorch modelini tamamladınız:

| Adım | Kullanılan Araç |
|------|-----------------|
| Veri hazırlama | `torch.arange`, `unsqueeze` |
| Eğitim/test bölme | Dilimleme (slicing) |
| Model tanımlama | `nn.Module`, `nn.Linear` |
| Kayıp hesabı | `nn.L1Loss` |
| Optimize etme | `torch.optim.SGD` |
| Kaydetme/yükleme | `torch.save`, `torch.load`, `state_dict` |

**Sonraki Modül →** Modül 02: Tüm iş akışını biraz farklı bir veri seti ve daha düzenli
yapıyla tekrar uygulayacağız.
